In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Tesla T4, 15360 MiB


In [2]:
import subprocess, sys

def pip_install(*specs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *specs], check=True)

pip_install("transformers==4.46.*", "accelerate==1.1.*", "bitsandbytes==0.49.2")
print("done")

done


In [3]:
import gc, torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

def load(dtype: str):
    if dtype == "fp16":
        return AutoModelForCausalLM.from_pretrained(
            MODEL, torch_dtype=torch.float16, device_map="cuda")
    if dtype == "int8":
        qc = BitsAndBytesConfig(load_in_8bit=True)
        return AutoModelForCausalLM.from_pretrained(
            MODEL, quantization_config=qc, device_map="cuda")
    if dtype == "int4":
        qc = BitsAndBytesConfig(load_in_4bit=True)
        return AutoModelForCausalLM.from_pretrained(
            MODEL, quantization_config=qc, device_map="cuda")
    raise ValueError(dtype)

def resident_vram_gb() -> float:
    torch.cuda.synchronize()
    return torch.cuda.memory_reserved() / (1024 ** 3)

In [4]:
def unload(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()

for dtype in ["fp16", "int8", "int4"]:
    model = load(dtype)
    vram = resident_vram_gb()
    print(dtype, vram)
    unload(model)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

fp16 3.060546875
int8 4.791015625
int4 2.87890625


In [5]:
print(torch.cuda.memory_allocated() / 1e9)

1.214033408


In [1]:
import subprocess, sys
def pip_install(*specs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *specs], check=True)
pip_install("transformers==4.46.*", "accelerate==1.1.*", "bitsandbytes==0.49.2")

In [2]:
import gc, torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

def load(dtype: str):
    if dtype == "fp16":
        return AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="cuda")
    if dtype == "int8":
        qc = BitsAndBytesConfig(load_in_8bit=True)
        return AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=qc, device_map="cuda")
    if dtype == "int4":
        qc = BitsAndBytesConfig(load_in_4bit=True)
        return AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=qc, device_map="cuda")
    raise ValueError(dtype)

def resident_vram_gb() -> float:
    torch.cuda.synchronize()
    return torch.cuda.memory_reserved() / (1024 ** 3)

In [3]:
for dtype in ["fp16", "int8", "int4"]:
    model = load(dtype)
    vram = resident_vram_gb()
    print(dtype, vram)
    del model
    gc.collect()
    torch.cuda.empty_cache()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


fp16 3.060546875
int8 1.7421875
int4 1.150390625


In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Tesla T4, 15360 MiB


In [2]:
import subprocess, sys

def pip_install(*specs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *specs], check=True)

pip_install("transformers==4.46.*", "accelerate==1.1.*", "bitsandbytes==0.49.2")
print("done")

done


In [3]:
import gc, torch
from transformers import AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

def reserved_mb() -> float:
    torch.cuda.synchronize()
    return torch.cuda.memory_reserved() / (1024 ** 2)

def unload(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()

In [4]:
samples = []
for i in range(5):
    model = AutoModelForCausalLM.from_pretrained(
        MODEL, torch_dtype=torch.float16, device_map="cuda")
    after_load = reserved_mb()
    unload(model)
    after_unload = reserved_mb()
    samples.append({"cycle": i, "after_load_mb": round(after_load, 1),
                     "after_unload_mb": round(after_unload, 1)})
    print(samples[-1])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


{'cycle': 0, 'after_load_mb': 3134.0, 'after_unload_mb': 3134.0}
{'cycle': 1, 'after_load_mb': 6268.0, 'after_unload_mb': 3138.0}
{'cycle': 2, 'after_load_mb': 6268.0, 'after_unload_mb': 3134.0}
{'cycle': 3, 'after_load_mb': 6268.0, 'after_unload_mb': 3138.0}
{'cycle': 4, 'after_load_mb': 6268.0, 'after_unload_mb': 3134.0}


In [5]:
tok_ids = torch.randint(0, 1000, (1, 64)).to("cuda")
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="cuda")

leaked_outputs = []
leak_samples = []
for i in range(20):
    out = model(tok_ids)
    leaked_outputs.append(out.logits)
    leak_samples.append({"iter": i, "reserved_mb": round(reserved_mb(), 1)})
    if i % 5 == 0:
        print(leak_samples[-1])

{'iter': 0, 'reserved_mb': 6272.0}
{'iter': 5, 'reserved_mb': 6612.0}
{'iter': 10, 'reserved_mb': 6952.0}
{'iter': 15, 'reserved_mb': 7292.0}


In [6]:
import numpy as np

def detect_leak(samples_mb, slope_threshold_mb_per_iter=1.0):
    x = np.arange(len(samples_mb))
    y = np.array(samples_mb)
    slope, intercept = np.polyfit(x, y, 1)
    leaking = slope > slope_threshold_mb_per_iter
    return {
        "slope_mb_per_iter": round(float(slope), 3),
        "threshold_mb_per_iter": slope_threshold_mb_per_iter,
        "leaking": bool(leaking),
        "n_samples": len(samples_mb),
    }

leak_result = detect_leak([s["reserved_mb"] for s in leak_samples])
print(leak_result)
assert leak_result["leaking"], "expected the Step 2 loop to be flagged as leaking"

{'slope_mb_per_iter': 68.0, 'threshold_mb_per_iter': 1.0, 'leaking': True, 'n_samples': 20}


In [7]:
unload(model)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="cuda")

fixed_samples = []
with torch.no_grad():
    for i in range(20):
        out = model(tok_ids)
        _ = out.logits.sum().item()
        fixed_samples.append({"iter": i, "reserved_mb": round(reserved_mb(), 1)})

fixed_result = detect_leak([s["reserved_mb"] for s in fixed_samples])
print(fixed_result)
assert not fixed_result["leaking"], "still leaking after the fix -- check both causes were removed"

{'slope_mb_per_iter': 0.286, 'threshold_mb_per_iter': 1.0, 'leaking': False, 'n_samples': 20}


In [8]:
import json
report = {
    "reload_loop_baseline": samples,
    "leaky_run": leak_result,
    "fixed_run": fixed_result,
    "leaky_samples": [s["reserved_mb"] for s in leak_samples],
    "fixed_samples": [s["reserved_mb"] for s in fixed_samples],
}
with open("leak_report.json", "w") as f:
    json.dump(report, f, indent=2)
print("wrote leak_report.json")

wrote leak_report.json


In [9]:
import json, os

class _Stop(Exception):
    pass

def fail(reason):
    print("GREEN CHECK: FAIL (%s)" % reason)
    raise _Stop()

def least_squares_slope(y):
    n = len(y)
    x = list(range(n))
    mean_x = sum(x) / n
    mean_y = sum(y) / n
    num = sum((x[i] - mean_x) * (y[i] - mean_y) for i in range(n))
    den = sum((x[i] - mean_x) ** 2 for i in range(n))
    return num / den if den != 0 else 0.0

def main():
    if not os.path.exists("leak_report.json"):
        fail("leak_report.json not found")
    with open("leak_report.json") as fh:
        report = json.load(fh)

    for key in ("leaky_samples", "fixed_samples", "leaky_run", "fixed_run"):
        if key not in report:
            fail("leak_report.json missing key: %s" % key)

    leaky_samples = report["leaky_samples"]
    fixed_samples = report["fixed_samples"]

    if len(leaky_samples) < 15:
        fail("leaky_samples has only %d points, need at least 15" % len(leaky_samples))
    if len(fixed_samples) < 15:
        fail("fixed_samples has only %d points, need at least 15" % len(fixed_samples))

    refit_leaky_slope = least_squares_slope(leaky_samples)
    refit_fixed_slope = least_squares_slope(fixed_samples)

    reported_leaky_slope = report["leaky_run"].get("slope_mb_per_iter")
    reported_fixed_slope = report["fixed_run"].get("slope_mb_per_iter")

    if reported_leaky_slope is None or reported_fixed_slope is None:
        fail("leaky_run or fixed_run missing slope_mb_per_iter")

    if abs(refit_leaky_slope - reported_leaky_slope) > 0.15 * abs(refit_leaky_slope):
        fail("refit leaky slope %.3f does not match reported %.3f within 15%%" %
             (refit_leaky_slope, reported_leaky_slope))

    if refit_leaky_slope < 5.0:
        fail("leaky run slope %.3f MB/iter is under the 5.0 MB/iter leak bar" %
             refit_leaky_slope)

    if refit_fixed_slope >= 1.0:
        fail("fixed run slope %.3f MB/iter is not flat (>= 1.0 MB/iter)" %
             refit_fixed_slope)

    print("refit leaky slope: %.3f MB/iter (reported: %.3f)" %
          (refit_leaky_slope, reported_leaky_slope))
    print("refit fixed slope: %.3f MB/iter (reported: %.3f)" %
          (refit_fixed_slope, reported_fixed_slope))
    print("GREEN CHECK: PASS")

try:
    main()
except _Stop:
    pass

refit leaky slope: 68.000 MB/iter (reported: 68.000)
refit fixed slope: 0.286 MB/iter (reported: 0.286)
GREEN CHECK: PASS


In [10]:
from google.colab import files
files.download("leak_report.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>